In [ ]:
# ============================================================
# Batman Meta Ranking Creators - NLP + K-Means
# ============================================================

# Cell 1: Install packages (run once if needed)
# !pip install pandas numpy scikit-learn matplotlib seaborn nltk


# ============================================================
# Cell 2: Import libraries
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

import re
import nltk

nltk.download("stopwords")

from nltk.corpus import stopwords


# ============================================================
# Cell 3: Load dataset
# ============================================================

file_path = "batman_meta_ranking_creators_sorted.csv"

df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)
display(df.head())

print("\nColumns:")
print(df.columns.tolist())


# ============================================================
# Cell 4: Basic dataset inspection
# ============================================================

print("Missing values:")
display(df.isnull().sum())

print("\nData types:")
display(df.dtypes)

print("\nDuplicate rows:", df.duplicated().sum())


# ============================================================
# Cell 5: Identify text columns
# ============================================================

text_columns = df.select_dtypes(include=["object"]).columns.tolist()

print("Text columns:")
print(text_columns)


# ============================================================
# Cell 6: Combine text columns for NLP
# ============================================================

# Replace missing values with empty strings
for col in text_columns:
    df[col] = df[col].fillna("")

# Combine all text columns into one NLP field
df["combined_text"] = df[text_columns].astype(str).agg(" ".join, axis=1)

display(df[["combined_text"]].head())


# ============================================================
# Cell 7: Text cleaning
# ============================================================

stop_words = set(stopwords.words("english"))

def clean_text(text):
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r"http\S+|www\S+|https\S+", " ", text)
    
    # Remove punctuation/numbers
    text = re.sub(r"[^a-z\s]", " ", text)
    
    # Remove extra whitespace
    text = re.sub(r"\s+", " ", text).strip()
    
    # Remove stopwords
    words = [
        word for word in text.split()
        if word not in stop_words and len(word) > 2
    ]
    
    return " ".join(words)

df["clean_text"] = df["combined_text"].apply(clean_text)

display(df[["combined_text", "clean_text"]].head())


# ============================================================
# Cell 8: TF-IDF NLP representation
# ============================================================

vectorizer = TfidfVectorizer(
    max_features=2000,
    min_df=2,
    max_df=0.90,
    ngram_range=(1, 2),
    sublinear_tf=True
)

X = vectorizer.fit_transform(df["clean_text"])

print("TF-IDF matrix shape:", X.shape)


# ============================================================
# Cell 9: Find the best number of K-means clusters
# ============================================================

# Test several cluster values
k_values = range(2, 11)

silhouette_scores = []

for k in k_values:
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    
    labels = model.fit_predict(X)
    
    score = silhouette_score(X, labels)
    silhouette_scores.append(score)
    
    print(f"k={k}: silhouette score={score:.4f}")


# ============================================================
# Cell 10: Plot silhouette scores
# ============================================================

plt.figure(figsize=(8, 5))

plt.plot(
    list(k_values),
    silhouette_scores,
    marker="o"
)

plt.xlabel("Number of clusters (k)")
plt.ylabel("Silhouette score")
plt.title("Selecting the Number of K-Means Clusters")

plt.xticks(list(k_values))
plt.grid(True)

plt.show()


# ============================================================
# Cell 11: Select the best k
# ============================================================

best_k = list(k_values)[np.argmax(silhouette_scores)]

print("Best number of clusters:", best_k)
print(
    "Best silhouette score:",
    max(silhouette_scores)
)


# ============================================================
# Cell 12: Train final K-means model
# ============================================================

kmeans = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=10
)

df["cluster"] = kmeans.fit_predict(X)

print("Cluster distribution:")
display(df["cluster"].value_counts().sort_index())


# ============================================================
# Cell 13: Examine cluster contents
# ============================================================

for cluster_number in sorted(df["cluster"].unique()):
    
    print("\n" + "=" * 70)
    print(f"CLUSTER {cluster_number}")
    print("=" * 70)
    
    cluster_data = df[df["cluster"] == cluster_number]
    
    print("Number of records:", len(cluster_data))
    
    display(
        cluster_data.head(10)
    )


# ============================================================
# Cell 14: Find important words for each cluster
# ============================================================

terms = vectorizer.get_feature_names_out()
centers = kmeans.cluster_centers_

for cluster_number in range(best_k):
    
    # Get the highest TF-IDF terms for this cluster
    top_indices = centers[cluster_number].argsort()[::-1][:20]
    
    top_terms = [
        terms[index]
        for index in top_indices
    ]
    
    print(f"\nCluster {cluster_number}")
    print(", ".join(top_terms))


# ============================================================
# Cell 15: PCA dimensionality reduction
# ============================================================

# Convert TF-IDF data to two dimensions
pca = PCA(n_components=2, random_state=42)

X_pca = pca.fit_transform(X.toarray())

df["PCA1"] = X_pca[:, 0]
df["PCA2"] = X_pca[:, 1]

print(
    "Explained variance:",
    pca.explained_variance_ratio_
)


# ============================================================
# Cell 16: Visualise K-means clusters
# ============================================================

plt.figure(figsize=(10, 7))

sns.scatterplot(
    data=df,
    x="PCA1",
    y="PCA2",
    hue="cluster",
    palette="tab10",
    s=80
)

plt.title("Batman Creator NLP K-Means Clusters")
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")

plt.legend(title="Cluster")
plt.grid(True)

plt.show()


# ============================================================
# Cell 17: Cluster summary
# ============================================================

cluster_summary = (
    df.groupby("cluster")
      .size()
      .reset_index(name="number_of_creators")
)

display(cluster_summary)


# ============================================================
# Cell 18: If there is a ranking column, inspect it
# ============================================================

# Look for likely ranking/score columns
possible_ranking_columns = [
    col for col in df.columns
    if any(
        keyword in col.lower()
        for keyword in [
            "rank",
            "ranking",
            "score",
            "rating",
            "meta"
        ]
    )
]

print("Possible ranking/score columns:")
print(possible_ranking_columns)


# ============================================================
# Cell 19: Numerical cluster comparison
# ============================================================

numeric_columns = df.select_dtypes(
    include=np.number
).columns.tolist()

print("Numeric columns:")
print(numeric_columns)

if len(numeric_columns) > 0:
    
    cluster_numeric_summary = (
        df.groupby("cluster")[numeric_columns]
          .mean()
          .round(2)
    )
    
    display(cluster_numeric_summary)


# ============================================================
# Cell 20: Save results
# ============================================================

output_file = "batman_meta_ranking_creators_kmeans.csv"

df.to_csv(
    output_file,
    index=False
)

print(f"Saved clustered dataset to: {output_file}")


# ============================================================
# Cell 21: Save cluster keywords
# ============================================================

cluster_keywords = []

for cluster_number in range(best_k):
    
    top_indices = centers[cluster_number].argsort()[::-1][:20]
    
    top_terms = [
        terms[index]
        for index in top_indices
    ]
    
    cluster_keywords.append({
        "cluster": cluster_number,
        "top_keywords": ", ".join(top_terms)
    })

cluster_keywords_df = pd.DataFrame(cluster_keywords)

display(cluster_keywords_df)

cluster_keywords_df.to_csv(
    "batman_cluster_keywords.csv",
    index=False
)

print("Cluster keyword file saved.")